## Import Libraries

Import necessary libraries for data manipulation and regular expressions.

In [ ]:
import pandas as pd
import re

## Load Data

Load the raw renewal calls data from CSV file.

In [ ]:
df = pd.read_csv('../../data/raw/renewal_calls.csv')

## Set Display Options

Configure pandas to display all columns for better data inspection.

In [ ]:
pd.set_option('display.max_columns', None)

## Initial Data Inspection

Display the dataframe to get an overview of the data and its columns.

In [ ]:
df

## Remove Unnamed Columns

Remove any unnamed columns that may have been created during CSV import.

In [ ]:
# Using loc and a boolean mask
df = df.loc[:, ~df.columns.str.contains('^Unnamed', case=False)]

# OR using the drop method with a list comprehension
df.drop(df.columns[df.columns.str.contains('unnamed', case=False)], axis=1, inplace=True)


## Data After Column Removal

Display the dataframe after removing unnamed columns.

In [ ]:
df

## Import Custom Utilities

Add the project root to the Python path and import custom utility functions for data processing.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../..'))

## Convert Column Names to Snake Case

Standardize column names by converting them to snake_case format using custom utility function.

In [ ]:
from src import utils
df = utils.convert_columns_to_snake_case(df)

## Data After Column Name Standardization

Display the dataframe with standardized column names in snake_case.

In [ ]:
df

## Check for Duplicate Columns

Identify any columns that contain identical data values.

In [ ]:
duplicates = []
cols = df.columns.tolist()

for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        c1, c2 = cols[i], cols[j]
        if df[c1].astype(str).equals(df[c2].astype(str)):
            duplicates.append((c1, c2))

for c1, c2 in duplicates:
    print(f"{c1}  ==  {c2}")

## Check for Duplicate Rows

Count the number of duplicate rows in the dataset.

In [ ]:
duplicate_mask = df.duplicated()
duplicate_count = duplicate_mask.sum()
    
print(f"Total duplicate rows: {duplicate_count}")

## Remove Duplicate Rows

Remove duplicate rows from the dataframe to ensure data quality.

In [ ]:
df = df.drop_duplicates()

## Data Shape After Deduplication

Check the dimensions of the dataframe after removing duplicate rows.

In [ ]:
df.shape

## Check Data Types

Examine the data types of each column in the dataframe.

In [ ]:
df.dtypes

## Convert Data Types

Convert call_date column to datetime type and analysed_call to integer type.

In [ ]:
df['call_date'] = pd.to_datetime(df['call_date'])
df['analysed_call'] = df['analysed_call'].astype('Int64')

## Verify Data Types After Conversion

Check data types after converting call_date and analysed_call columns.

In [ ]:
df.dtypes

## Data After Type Conversion

Display the dataframe after converting data types.

In [ ]:
df

## Format Call Date

Format the call_date column to a consistent YYYY-MM-DD string format.

In [ ]:
df['call_date'] = pd.to_datetime(df['call_date'], dayfirst=True).dt.strftime('%Y-%m-%d')

## Data After Date Formatting

Display the dataframe after formatting the call_date column.

In [ ]:
df

## Check for Missing Values

Count the number of missing values in each column.

In [ ]:
df.isnull().sum()

## Remove Rows with Missing Customer Reference

Remove rows where co_ref (customer reference) is missing as it's a key identifier.

In [ ]:
df = df[df['co_ref'].notna()]

## Verify Missing Values After Removal

Check missing values after removing rows with missing customer reference.

In [ ]:
df.isnull().sum()

## Data After Removing Missing Customer References

Display the dataframe after removing rows with missing customer reference.

In [ ]:
df

## Standardize Call Direction Values

Standardize call_direction values to consistent 'OUT_BOUND' and 'IN_BOUND' format.

In [ ]:
df['call_direction'] = df['call_direction'].replace({
    'Outbound': 'OUT_BOUND',
    'Inbound': 'IN_BOUND'
})
df['call_direction'].value_counts()

## Data After Call Direction Standardization

Display the dataframe after standardizing call_direction values.

In [ ]:
df

## Check Missing Values After Standardization

Check for missing values after standardizing call_direction values.

In [ ]:
df.isnull().sum()

## Check Unique Values per Column

Count the number of unique values in each column.

In [ ]:
df.nunique()

## Detailed Value Counts for Each Column

Display value counts for each column, including null values, to understand data distribution.

In [ ]:
for col in df.columns:
    print(f"Column: {col}")
    print(df[col].value_counts(dropna=False))
    print("-" * 40)

## Identify Columns with High Null Percentage

Identify columns with more than 80% null values that may need to be removed.

In [ ]:
null_percentages = df.isnull().mean()
cols_to_drop_candidates = null_percentages[null_percentages > 0.80].index.tolist()
print('Columns with >80% null fraction:')
print(cols_to_drop_candidates)

## Drop High-Null and Complex Columns

Remove columns with high null percentages and complex categorical columns that may not be useful for modeling.

In [ ]:
# Drop the specific high-null / high-complexity columns identified from the dataset
cols_to_drop_candidates = [col for col in cols_to_drop_candidates if col in df.columns]
df.drop(columns=cols_to_drop_candidates, inplace=True)

complex_columns_to_drop = [
    'agent_renewal_pitch_category',
    'customer_renewal_response_category',
    'agent_response_category',
    'analysed_call',
    'call_year',
]
complex_columns_to_drop = [col for col in complex_columns_to_drop if col in df.columns]
df.drop(columns=complex_columns_to_drop, inplace=True)

## Data Shape After Column Removal

Check the dimensions of the dataframe after removing high-null and complex columns.

In [ ]:
df.shape

## Standardize Categorical Column Values

Standardize values in categorical columns to consistent formats like Yes/No/Not Discussed/Unknown.

In [ ]:
for col in ['percentage_price_increase_mentioned','monetary_price_increase_mentioned','customer_asked_for_justification','discount_offered']:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    df[col] = df[col].apply(
        lambda x: "Yes" if str(x).strip().lower() == "yes"
        else ("No" if str(x).strip().lower() == "no"
        else ("Not Discussed" if "not discussed" in str(x).strip().lower()
        else "Unknown"))
    )
    print(df[col].value_counts(dropna=False))

for col in ['membership_renewal_decision','serious_complaint','other_complaint','discussion_on_price_increase','renewal_impact_due_to_price_increase','discount_or_waiver_requested','call_reschedule_request','agent_flagged_membership_status_alert','agent_renewal_initiation','explicit_competitor_mention','explicit_switching_intent','mentioned_competitors','price_switching_mentioned']:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    df[col] = df[col].apply(lambda x: "Yes" if x == "yes" else ("No" if x == "no" else "Unknown"))
    print(df[col].value_counts(dropna=False))

df['competitor_value_comparison'] = df['competitor_value_comparison'].apply(
    lambda x: "Not Discussed" if  "not discussed" in str(x).strip().lower()
    else ("Similar Value" if str(x).strip().lower() == "similar value"
    else ("Better Value" if str(x).strip().lower() == "better value"
    else ("Not Applicable" if "not applicable" in str(x).strip().lower()
    else "Unknown")))
)
print(df['competitor_value_comparison'].value_counts(dropna=False))

df['competitor_benefits_mentioned'] = df['competitor_benefits_mentioned'].apply(
    lambda x: "Not Discussed" if "not discussed" in str(x).strip().lower()
    else ("Discounts" if "discount" in str(x).strip().lower()
    else ("Better Service" if "better service" in str(x).strip().lower()
    else ("Not Applicable" if "not applicable" in str(x).strip().lower()
    else "Unknown")))
)
print(df['competitor_benefits_mentioned'].value_counts(dropna=False))

df['topic_introduced_by'] = df['topic_introduced_by'].apply(
    lambda x: "Not Discussed" if "not discussed" in str(x).strip().lower()
    else ("Agent" if str(x).strip().lower() == "agent"
    else ("Customer" if str(x).strip().lower() == "customer"
    else "Unknown"))
)
print(df['topic_introduced_by'].value_counts(dropna=False))

def process_price_range(x):
    # Handle NaN and Not Discussed
    if "not discussed" in str(x).lower():
        return "Not Discussed"
    
    # Extract numbers
    numbers = re.findall(r'\d+\.?\d*', str(x))
    
    if len(numbers) >= 2:
        nums = list(map(float, numbers[:2]))
        return sum(nums) / 2
    
    return "Unknown"
df['price_range_mentioned'] = df['price_range_mentioned'].apply(process_price_range)

df['customer_response'] = df['customer_response'].apply(
    lambda x: "Not Discussed" if "not discussed" in str(x).strip().lower()
    else ("Positive" if str(x).strip().lower() == "positive"
    else ("Negative" if str(x).strip().lower() == "negative"
    else ("Neutral" if str(x).strip().lower() == "neutral"
    else "Unknown")))
)
print(df['customer_response'].value_counts(dropna=False))

df['desire_to_cancel'] = df['desire_to_cancel'].apply(
    lambda x: "Not Discussed" if "not discussed" in str(x).strip().lower()
    else ("Renewed" if "renew" in str(x).strip().lower()
    else ("Desired to Cancel" if "cancel" in str(x).strip().lower()
    else "Unknown"))
)
print(df['desire_to_cancel'].value_counts(dropna=False))

## Check Desire to Cancel Distribution

Display the distribution of desire_to_cancel values after standardization.

In [ ]:
(df['desire_to_cancel'].value_counts(dropna=False))

## Check Missing Values After Standardization

Check for missing values after standardizing categorical column values.

In [ ]:
df.isnull().sum()

## Data Shape After Standardization

Check the dimensions of the dataframe after standardizing categorical values.

In [ ]:
df.shape

## Final Cleaned Data

Display the final cleaned and processed dataframe.

In [ ]:
df

## Save Cleaned Data

Save the cleaned renewal calls data to the interim data directory as CSV.

In [ ]:
df.to_csv('../../data/interim/renewal_calls_cleaned.csv', index=False)